In [1]:
#| echo: true
#| code-fold: true
import time

import numpy as np
import torch
import torch.nn as nn

torch.manual_seed(6600)
print(f"torch {torch.__version__}   cuda available: {torch.cuda.is_available()}")

torch 2.7.1+cu126   cuda available: True


# PyTorch and Training

- Thursdays, 3:30-6:00 PM · ICC 103
- Weeks 5 and 6 of 14 · Sep 24 and Oct 1
- **Quiz 4 is at the end of class today**, covering Week 4 and Lab 3

## Agenda {.smaller}

Backpropagation is finished, apart from the vanishing-gradient cures we listed
but did not work through. Those are item 7. Everything else is the tooling
that makes backprop usable:

1. **Tensors**: what PyTorch adds to a NumPy array
2. **Autograd**: `.backward()`, `.grad`, and why gradients accumulate
3. **Models**: `nn.Linear`, `nn.Module`, and which loss goes with which task
4. **The first training loop**: Lab 2's network, one backward pass per step
5. **Data**: `Dataset`, `DataLoader`, and what batch size actually costs you
6. **Optimizers**: momentum, RMSProp, Adam
7. **Schedules, initialization, regularization**
8. **A complete training script**

Spotlights first. Quiz 4 in the last 20 minutes.

::: {.callout-note}
## Pace
Items 1 through 4 are the core and come first. Items 5 onward are the
decisions that surround training, and they are written to be readable on your
own if we run short.
:::

## Where backpropagation left us

Three weeks of work to arrive at one fact: **the gradient is cheap now.**

| | Lab 2 | After backpropagation |
|---|---|---|
| Gradient from | finite differences | reverse-mode autodiff |
| Work per step | 50 forward passes | 1 forward + 1 backward |
| 400 steps | 20,001 forward passes | 401 forward passes |
| Exact? | no, $O(\varepsilon)$ error | yes, to floating point |

That ratio was 50 for a 49-parameter network. For a 25-million-parameter
network it is 25 million.

Everything that can still go wrong is now about the **optimizer**, the
**data**, the **initialization**, and the **loss surface**. That list is the
rest of this deck.

## Cures for vanishing and exploding gradients {.smaller}

We ended last week on the problem and **named** the cures without working
through any of them. Each one lands somewhere today:

| Cure | Where it is in this deck |
|---|---|
| **Weight initialization** | Initialization: Xavier and He |
| **Gradient clipping** | Learning-rate schedules |
| **Batch normalization** | Regularization |
| **Better activations**, ReLU over sigmoid | Week 2, already yours |
| **Skip connections** | With CNNs, next week |

They come after the PyTorch foundation rather than before it, and that
ordering is deliberate: every one of them is a line of PyTorch you cannot
read yet. `nn.init.kaiming_normal_` means nothing until `nn.Linear` does.

::: {.callout-note}
## Initialization is the one to remember
Initialization is the cure that decides whether a deep network trains **at
all**, and the measured version of that claim is later in this deck.
:::

## What changes, and what does not

You have written a backward pass by hand, in Lab 3. Nothing in PyTorch is
doing anything you have not done.

:::: {.columns .contrast}
::: {.column width="48%" .col-yes}
### Already yours
- The chain rule on a graph
- Local derivatives at a node
- Adding gradients where paths meet
- Reverse topological order
- The three layer gradients
:::
::: {.column width="48%" .col-no}
### What PyTorch adds
- It records the graph for you
- It runs on a GPU
- It has the layers already written
- It has the optimizers already written
- It is fast enough to be useful
:::
::::

The point of Lab 3 was that `.backward()` should be **boring** by the time you
first call it.

# Tensors

- A tensor is an array that can also record how it was computed
- Most first-week PyTorch errors are `dtype` or `device`

## Rank, shape, and what a tensor is

A **tensor** is an $n$-dimensional array. **Rank**, or `ndim`, is the number of
axes, meaning the number of indices needed to reach one element.

| Rank | Shape | What it holds |
|---|---|---|
| 0 | `[]` | a scalar, such as a cost |
| 1 | `[d]` | one feature vector |
| 2 | `[N, d]` | a batch of feature vectors |
| 3 | `[N, T, d]` | a batch of sequences |
| 4 | `[N, C, H, W]` | a batch of images |

A PyTorch tensor is a NumPy array plus three things Lab 2 did not have:

- A **device**, CPU or CUDA or MPS
- A **`requires_grad`** flag
- A recorded **graph**, so it knows which operation produced it

## Tensors and NumPy arrays

The arithmetic is the same. `torch.from_numpy` even shares memory on CPU, so
writing through one view changes the other.

In [2]:
#| echo: true
A_np = np.arange(6).reshape(2, 3)
A = torch.from_numpy(A_np)          # shares storage with A_np on CPU

print(f"numpy  {A_np.shape}  torch  {tuple(A.shape)}  ndim {A.ndim}  "
      f"numel {A.numel()}")

A_np[0, 0] = 99
print(f"wrote through the numpy view; torch sees {A[0, 0].item()}")

x = torch.tensor([[0.7, -1.2]])                  # one batched example, (1, 2)
W = torch.randn(3, 2, requires_grad=True)        # (units, inputs), as in Lab 2
b = torch.zeros(3, requires_grad=True)
z = x @ W.T + b                                  # the batched layer from Lab 2

print(f"\nx {tuple(x.shape)} @ W.T {tuple(W.T.shape)} + b {tuple(b.shape)} "
      f"-> z {tuple(z.shape)}")
print(f"z was produced by {type(z.grad_fn).__name__}, "
      f"which is the graph autograd recorded")

numpy  (2, 3)  torch  (2, 3)  ndim 2  numel 6
wrote through the numpy view; torch sees 99

x (1, 2) @ W.T (2, 3) + b (3,) -> z (1, 3)
z was produced by AddBackward0, which is the graph autograd recorded


## dtype and device {.smaller}

Two attributes that cause most first-week PyTorch errors.

:::: {.columns}
::: {.column width="52%"}
**dtype.** Memory is `numel` times bytes per element.

| dtype | Bytes | Used for |
|---|---|---|
| `float32` | 4 | the default for training |
| `float64` | 8 | precision, and slower |
| `float16` | 2 | fast GPU training |
| `bfloat16` | 2 | wider exponent, stabler |
| `int64` | 8 | indices, class labels |
:::
::: {.column width="48%"}
**device.** Tensors must be on the same device to interact.

```python
torch.cuda.is_available()
x = x.to("cuda")
```

`RuntimeError: Expected all tensors to be on the same device` is the error you
will hit first. The fix is always to move one of them.

`nn.CrossEntropyLoss` wants labels as `int64`. That is the other one.
:::
::::

## Shape discipline

The habit that saves the most debugging time is simple. **Print shapes, not
values.**

```python
print(x.shape, W.shape, z.shape)
```

These two problems look the same from the outside but are not. A network that
trains badly is a research problem. A network with the wrong shapes is a bug,
and usually a quiet one: broadcasting will turn a `(32, 1)` against a `(32,)`
into a `(32, 32)` without complaining, and hand you a cost that is
arithmetically correct and completely meaningless.

::: {.callout-important}
## The most common shape bug
`criterion(pred, y)` where `pred` is `(N, 1)` and `y` is `(N,)`. This
broadcasts into an $N \times N$ matrix of differences, the cost comes out
plausible, and the model never learns. Use `y.unsqueeze(1)` or
`pred.squeeze(1)` and make the shapes match deliberately.
:::

## Broadcasting, drawn

[![](images/fig-broadcasting.png){width=1000 .dw92}](images/fig-broadcasting.png){target="_blank" .zoom}

Broadcasting stretches a size-1 or missing dimension to match the other
operand. All three of these are the **same rule**. Two of them are the reason
the rule exists. The third is a $(4,1)$ minus a $(4,)$: the batch dimension
lines up against the feature dimension, every prediction gets compared against
every label, and you get a $4 \times 4$ where you wanted a $4 \times 1$.

## Reading the shape bug {.codetight}

Nothing raises. The cost comes out $9.7\times$ too large and still looks
exactly like a cost.

In [3]:
#| echo: true
pred = torch.tensor([[0.9], [0.2], [0.7], [0.1]])   # (4, 1)  model output
y    = torch.tensor([1.0, 0.0, 1.0, 0.0])           # (4,)    labels

diff = pred - y                       # no warning, no error, wrong shape
print(f"(4, 1) - (4,)   ->   {tuple(diff.shape)}")
print()
print(f"  broadcast  MSE  {(diff ** 2).mean().item():.4f}")
print(f"  correct    MSE  {((pred - y.unsqueeze(1)) ** 2).mean().item():.4f}")

(4, 1) - (4,)   ->   (4, 4)

  broadcast  MSE  0.3625
  correct    MSE  0.0375


`nn.MSELoss` does print a `UserWarning` for this exact mismatch, which is a
real safety net. But the subtraction one line earlier was already silent, so
any loss you write yourself gets no such warning, and a warning that scrolls
past in cell 40 of a notebook is not a warning anyone reads.

::: {.callout-tip}
## Check the shapes
`assert pred.shape == y.shape` on the line before the loss. It costs nothing
and it closes this off permanently.
:::

# Autograd

- The backward pass you wrote in Lab 3, with a bigger graph and a GPU
- Three rules that explain most of autograd, and one common mistake

## `requires_grad` and the recorded graph

Three rules that explain most of autograd's behaviour:

1. An operation is recorded if **any** input has `requires_grad=True`
2. `.backward()` is called on a **scalar**, and it fills `.grad` on every leaf
   tensor that requested gradients
3. Intermediate results do **not** keep their gradient unless you ask, with
   `retain_grad()`

Rule 2 is the reason reverse mode is the right choice, restated as an API
constraint: the thing you differentiate has to be one number.

To stop recording, either wrap the block in `torch.no_grad()` or call
`.detach()`. Do this for evaluation and inference: it is faster, and it does
not build a graph you are about to throw away.

```python
with torch.no_grad():
    val_cost = criterion(model(x_val), y_val)
```

## Reading gradients out of `.grad` {.codetight}

The graph from **Lab 3, part 2c**, handed to autograd instead of to your
`Value` class. Same three inputs, same expected answers.

In [4]:
#| echo: true
A = torch.tensor([2.0], requires_grad=True)
B = torch.tensor([3.0], requires_grad=True)
D = torch.tensor([5.0], requires_grad=True)

C = A * B;   C.retain_grad()
F = D + C;   F.retain_grad()
G = F + A;   G.retain_grad()
H = G * B

H.backward()

print(f"forward   C={C.item():4.0f}  F={F.item():4.0f}  "
      f"G={G.item():4.0f}  H={H.item():4.0f}")
print()
for name, t in [("G", G), ("F", F), ("C", C), ("D", D), ("B", B), ("A", A)]:
    print(f"  dH/d{name}  {t.grad.item():>6.0f}")

forward   C=   6  F=  11  G=  13  H=  39

  dH/dG       3
  dH/dF       3
  dH/dC       3
  dH/dD       3
  dH/dB      19
  dH/dA      12


Three lines of bookkeeping replaced the whole engine you wrote. Look at the
answer for $A$: **12**, not 3 and not 9. $A$ reaches $H$ along two paths, and
autograd **added** the two contributions, which is the same rule your
`backward` had to implement by hand.

## Gradient accumulation into `.grad`

`.grad` is a **running sum**, for exactly the reason we saw at the fork in the
graph: a node may be reached along more than one path, so the backward pass
adds rather than overwrites.

Here is the failure mode rather than a warning about it.

In [5]:
#| echo: true
w = torch.tensor([2.0], requires_grad=True)
x, y = torch.tensor([3.0]), torch.tensor([12.0])

for call in (1, 2, 3):
    loss = (w * x - y) ** 2
    loss.backward()
    print(f"backward call {call}:  w.grad = {w.grad.item():>7.1f}"
          f"   ({call}x the true gradient)")

w.grad.zero_()
loss = (w * x - y) ** 2
loss.backward()
print(f"\nafter zero_():   w.grad = {w.grad.item():>7.1f}   (correct again)")

backward call 1:  w.grad =   -36.0   (1x the true gradient)
backward call 2:  w.grad =   -72.0   (2x the true gradient)
backward call 3:  w.grad =  -108.0   (3x the true gradient)

after zero_():   w.grad =   -36.0   (correct again)


## `zero_grad` in the training loop

Every training step is these four lines, and the order matters:

```python
optimizer.zero_grad()    # clear the running sum from last step
loss = criterion(model(x), y)
loss.backward()          # write .grad on every parameter
optimizer.step()         # theta <- theta - lr * theta.grad
```

Forget `zero_grad()` and step 3 adds to whatever was already there. You are
not descending on the current gradient; you are descending on the sum of every
gradient since the model was built. The cost usually diverges, and **it looks
exactly like a learning rate that is too high.**

::: {.callout-note}
## The accumulation is a feature too
Deliberately skipping `zero_grad()` for a few mini-batches simulates a larger
batch than fits in memory. That is **gradient accumulation**, and it is a real
technique used to train large models on small hardware. It is only a bug when
it is an accident.
:::

# Models

- `nn.Linear` is Lab 2's layer
- `nn.Module` is a forward pass with parameters attached

## `nn.Linear` and the shape convention

`nn.Linear(in_features, out_features)` stores $\mathbf{W}$ as
`(out_features, in_features)` and $\mathbf{b}$ as `(out_features,)`. That is
the `(units, inputs)` convention from Week 2, unchanged, which is why the
batched form carries a transpose.

In [6]:
#| echo: true
torch.manual_seed(6600)
layer = nn.Linear(2, 3)
x = torch.randn(5, 2)                       # five examples, two features

print(f"layer.weight {tuple(layer.weight.shape)}   "
      f"layer.bias {tuple(layer.bias.shape)}")
print(f"x {tuple(x.shape)}  ->  layer(x) {tuple(layer(x).shape)}")

# the same arithmetic, written out
by_hand = x @ layer.weight.T + layer.bias
print(f"\nmax |layer(x) - (x @ W.T + b)| = "
      f"{(layer(x) - by_hand).abs().max().item():.2e}")
print("nn.Linear is that line, plus the gradient tape")

layer.weight (3, 2)   layer.bias (3,)
x (5, 2)  ->  layer(x) (5, 3)

max |layer(x) - (x @ W.T + b)| = 0.00e+00
nn.Linear is that line, plus the gradient tape


## `nn.Module`

A model is a class with two jobs: declare the layers in `__init__`, and write
the forward pass in `forward`. Subclassing `nn.Module` is what **registers**
the parameters, so the optimizer, `.to(device)` and saving all find them.

In [7]:
#| echo: true
class MLP(nn.Module):
    def __init__(self, hidden=16):
        super().__init__()
        self.hidden = nn.Linear(1, hidden)
        self.out = nn.Linear(hidden, 1)

    def forward(self, x):
        return self.out(torch.tanh(self.hidden(x)))


model = MLP()
print(model)
print()
for name, p in model.named_parameters():
    print(f"  {name:<14} {str(tuple(p.shape)):>10}  {p.numel():>3} "
          f"parameter{'s' if p.numel() != 1 else ''}")
print(f"\ntotal: {sum(p.numel() for p in model.parameters())} "
      f"(Lab 2's network, exactly)")

MLP(
  (hidden): Linear(in_features=1, out_features=16, bias=True)
  (out): Linear(in_features=16, out_features=1, bias=True)
)

  hidden.weight     (16, 1)   16 parameters
  hidden.bias         (16,)   16 parameters
  out.weight        (1, 16)   16 parameters
  out.bias             (1,)    1 parameter

total: 49 (Lab 2's network, exactly)


## Calling a model, and train and eval mode

Call `model(x)`, never `model.forward(x)`. The first runs registered hooks and
respects train and eval mode; the second skips them.

- **`nn.Sequential`** is the same thing when the graph is a straight line
- **`model.train()`** and **`model.eval()`** change the behaviour of dropout
  and batch normalization. Neither is in this model yet, so neither matters
  yet, and both will bite you later in this deck
- **`model.parameters()`** is what you hand the optimizer

```python
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
```

::: {.callout-important}
## Forgetting `model.eval()`
Your validation cost will be computed with dropout still switched on and batch
norm still updating its running statistics. The number will be worse than the
truth, noisier than the truth, and it will change if you evaluate twice.
:::

## Losses, and what goes in the final layer

| Task | Final layer | PyTorch loss |
|---|---|---|
| Regression | linear, 1 unit | `nn.MSELoss` |
| Binary classification | linear, 1 unit | `nn.BCEWithLogitsLoss` |
| Multi-class, $C$ classes | linear, $C$ units | `nn.CrossEntropyLoss` |

Note what is **not** in that middle column: no sigmoid, and no softmax.

`nn.BCELoss` exists and takes probabilities, so it needs a sigmoid in the
model. Prefer **`BCEWithLogitsLoss`**, which folds the sigmoid in and is
numerically stabler for exactly the cancellation reason from Week 4: computing
$\hat{y} - y$ directly avoids ever forming $\log \hat{p}$ when $\hat{p}$ is
near zero.

## Logits and `CrossEntropyLoss`

In Keras you apply softmax in the model. **In PyTorch you do not.**
`nn.CrossEntropyLoss` applies $\log\mathrm{softmax}$ internally and then takes
the negative log-likelihood:

$$\mathcal{L}(z, y) = -\log \frac{e^{z_y}}{\sum_j e^{z_j}}
  \;=\; -z_y + \log \sum_j e^{z_j}$$

So classification models in this course return **logits**, raw scores of shape
`(N, C)`, and the targets are **integer class labels** of shape `(N,)` with
dtype `int64`. Not one-hot.

Apply softmax in the model as well and you have applied it twice: the gradients
degrade and the numerics get worse. It is a quiet bug, because the model still
trains, just badly.

## Why the softmax is folded in {.smaller}

Two reasons to keep the two steps together instead of computing a probability
and then taking its log.

**Numerics.** Written separately, a confident wrong prediction puts
$\hat{p}_y \approx 10^{-40}$ into a `log`, and in float32 that underflows to
zero and the loss returns `inf`. Written as $-z_y + \log\sum_j e^{z_j}$, the
sum can be rescaled by its own largest term before exponentiating, so nothing
ever overflows or underflows.

**The gradient collapses.** Compose the two and almost everything cancels:

$$\frac{\partial \mathcal{L}}{\partial z_i} = p_i - y_i,
  \qquad p = \mathrm{softmax}(z), \quad y \text{ one-hot}$$

Predicted minus actual. No softmax Jacobian, no division, no chain of
intermediate terms to lose precision through. It is the multi-class version of
the sigmoid-plus-BCE cancellation from Week 4, and it is the reason these
loss-and-activation pairings are not arbitrary.

::: {.callout-note}
## Read the gradient
If the true class has $p_y = 0.7$, its logit gets $0.7 - 1 = -0.3$: push it up.
Every other class gets $+p_i$: push it down, in proportion to how much
probability it stole.
:::

# The first training loop

- Lab 2's network, Lab 2's data, Lab 2's initialization
- One backward pass per step instead of fifty forward passes

## The five lines

These will not change for the rest of the semester:

```python
for step in range(STEPS):
    optimizer.zero_grad()               # clear the running sum
    loss = criterion(model(x), y)       # forward, and record
    loss.backward()                     # backward, fill every .grad
    optimizer.step()                    # theta <- theta - lr * grad
```

Everything else is which model, which loss, which optimizer, and what data
comes in. The optimizer is plain **SGD** here, the same update rule from
Week 1:

$$\theta^{(t+1)} = \theta^{(t)} - \eta\, \nabla_{\theta} \mathcal{L}\big(\theta^{(t)}\big)$$

Momentum, Adam and schedules are later in this deck. Right now we only want the
gradient to be correct and cheap.

## Lab 2's network, trained with autograd {.codetight}

Same seed, same weights, same 400 steps, same learning rate. Source code for
your reference. The printed costs should match Lab 2 to four decimals; the work
per step should not.

In [8]:
#| echo: true
SEED, HIDDEN, STEPS, LR = 6600, 16, 400, 0.1
SHAPES = [("W", (HIDDEN, 1)), ("b", (HIDDEN,)), ("W", (1, HIDDEN)), ("b", (1,))]

# Lab 2's data and initialization, reproduced exactly: two generators, both
# seeded with SEED, so neither advances the other's stream.
rng_data, rng_init = np.random.default_rng(SEED), np.random.default_rng(SEED)
x_np = np.linspace(-1, 1, 200).reshape(-1, 1)
y_np = np.sin(3.0 * x_np.ravel()) + rng_data.normal(0, 0.10, 200)

theta, i = np.zeros(sum(int(np.prod(s)) for _, s in SHAPES)), 0
for kind, shape in SHAPES:
    n = int(np.prod(shape))
    theta[i:i + n] = (rng_init.normal(0, 1 / np.sqrt(shape[1]), n)
                      if kind == "W" else 0.0)
    i += n

model = nn.Sequential(nn.Linear(1, HIDDEN), nn.Tanh(),
                      nn.Linear(HIDDEN, 1)).double()
with torch.no_grad():
    model[0].weight.copy_(torch.from_numpy(theta[:HIDDEN].reshape(HIDDEN, 1)))
    model[0].bias.copy_(torch.from_numpy(theta[HIDDEN:2 * HIDDEN]))
    model[2].weight.copy_(torch.from_numpy(
        theta[2 * HIDDEN:3 * HIDDEN].reshape(1, HIDDEN)))
    model[2].bias.copy_(torch.from_numpy(theta[-1:]))

x = torch.from_numpy(x_np)
y = torch.from_numpy(y_np).unsqueeze(1)
optimizer = torch.optim.SGD(model.parameters(), lr=LR)
criterion = nn.MSELoss()

start = float(criterion(model(x), y))
t0 = time.perf_counter()
for step in range(STEPS):
    optimizer.zero_grad()
    loss = criterion(model(x), y)
    loss.backward()
    optimizer.step()
elapsed = time.perf_counter() - t0
end = float(criterion(model(x), y))
n_params = sum(p.numel() for p in model.parameters())

print(f"parameters      {n_params}")
print(f"cost at start   {start:.4f}      (Lab 2: 1.3256)")
print(f"cost at end     {end:.4f}      (Lab 2: 0.0190)")
print(f"wall time       {elapsed:.2f} s for {STEPS} steps")
print(f"work per step   1 forward + 1 backward, not {n_params + 1} forwards")

parameters      49
cost at start   1.3256      (Lab 2: 1.3256)
cost at end     0.0190      (Lab 2: 0.0190)
wall time       0.18 s for 400 steps
work per step   1 forward + 1 backward, not 50 forwards


## What is left to get wrong {.smaller}

The gradient is solved. Everything in the rest of this deck is a way for a
**correct** gradient to still produce a bad model.

:::: {.columns}
::: {.column width="50%"}
**The optimizer**

- One learning rate for differently curved directions
- Local minima, saddle points, plateaus
- A rate that was right at step 1 and wrong at step 10,000
:::
::: {.column width="50%"}
**Everything else**

- Initialization that kills the gradient before training starts
- Batches too small to point the right way
- A model that fits the noise instead of the signal
:::
::::

That demo was 49 parameters on 200 clean points with one optimizer and no
regularization. Every one of those choices was made for you.

# Data

- Why anyone uses batches
- `Dataset`, `DataLoader`, and the splits

## Full batch, mini-batch, stochastic {.smaller}

One **optimizer step** is one parameter update. How much data goes into it has
a name. For $N = 100$ training examples:

| Paradigm | Batch size | Updates per epoch | Gradient noise |
|---|---|---|---|
| Full batch | 100 | 1 | none |
| Stochastic | 1 | 100 | very high |
| Mini-batch | 25 | 4 | moderate |

An **epoch** is one pass through the whole training set, which is
$\lceil N/B \rceil$ iterations.

The demo above was **full batch** on 200 points, so there was one update per
epoch and the cost curve came out smooth. That is not what anyone does, for two
reasons: the whole dataset usually does not fit in memory, and the noise turns
out to be **useful** rather than merely tolerable.

## Batch size and gradient noise, measured

[![](images/fig-batch-noise.png){width=1000 .dw89}](images/fig-batch-noise.png){target="_blank" .zoom}

A mini-batch gradient is an **unbiased but noisy** estimate of the full-batch
gradient. Measured on a real network: the relative error is $3.24$ at $B = 1$,
$0.60$ at $B = 32$, $0.21$ at $B = 256$, and $0.11$ at $B = 1024$, tracking
$1/\sqrt{B}$ almost exactly.

## What the $1/\sqrt{B}$ curve means in practice

Doubling the batch **doubles the work per step** and reduces the gradient error
by a factor of only $\sqrt{2}$, about 29%. Four times the compute buys you half
the noise.

Two consequences:

1. **Big batches have diminishing returns.** Going from 32 to 1024 is 32 times
   the compute per step for a 5.5-fold noise reduction
2. **Small batches are not just tolerable, they help.** The noise lets the
   optimizer rattle out of sharp local minima and saddle points. A full-batch
   run gets stuck in places a mini-batch run walks straight past

::: {.callout-note}
## How to pick a batch size
Pick the largest batch that fits comfortably in memory, then tune the learning
rate for it. Batch sizes are powers of two out of habit and memory alignment,
not mathematics.
:::

## `Dataset` {.smaller}

A `Dataset` is a class you write with three methods.

| Method | What it does |
|---|---|
| `__init__` | set up once: file paths, a dataframe, the transform list |
| `__len__` | return how many examples there are |
| `__getitem__(i)` | **produce example `i`** and return it as tensors |

`__getitem__` is the important one, and it is not a lookup. It is the recipe
for turning example $i$ into tensors, and it runs **every time that example is
used**. For a real dataset that means: open the file on disk, decode the JPEG
or parse the audio, apply the crop and flip, convert to a tensor, return it.

Below, the data is small enough to sit in memory, so `__getitem__` really is
just an index. That is the exception, not the rule.

In [9]:
#| echo: true
from torch.utils.data import DataLoader, Dataset, random_split


class SineData(Dataset):
    def __init__(self, n=200, seed=6600):
        rng = np.random.default_rng(seed)
        self.x = np.linspace(-1, 1, n, dtype=np.float32).reshape(-1, 1)
        self.y = (np.sin(3.0 * self.x.ravel())
                  + rng.normal(0, 0.10, n)).astype(np.float32).reshape(-1, 1)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, i):
        return torch.from_numpy(self.x[i]), torch.from_numpy(self.y[i])


data = SineData()
xi, yi = data[7]
print(f"len(data)     {len(data)}")
print(f"data[7]       x {tuple(xi.shape)} {xi.item():+.4f}   "
      f"y {tuple(yi.shape)} {yi.item():+.4f}")

len(data)     200
data[7]       x (1,) -0.9296   y (1,) -0.4883


## `DataLoader`

A `DataLoader` wraps a `Dataset` and hands you batches. Shuffling and the short
final batch are the easy part. The reason the class exists is **speed**.

One training step on a GPU takes a few milliseconds. Reading and decoding one
JPEG takes around ten. Do both in one process and the GPU sits idle most of the
run, waiting for the CPU to feed it. **Most slow training runs in practice are
slow for this reason**, not because the model is large.

A `DataLoader` avoids that by building the next batch **while the GPU is still
working on the current one**, in several processes at once.

In [10]:
#| echo: true
loader = DataLoader(data, batch_size=32, shuffle=True, drop_last=False)

print(f"{len(data)} examples, batch_size 32  ->  {len(loader)} batches")
for k, (xb, yb) in enumerate(loader):
    print(f"  batch {k}: x {tuple(xb.shape)}  y {tuple(yb.shape)}")

print("\nthe last batch holds 8, because 200 is not a multiple of 32")

200 examples, batch_size 32  ->  7 batches
  batch 0: x (32, 1)  y (32, 1)
  batch 1: x (32, 1)  y (32, 1)
  batch 2: x (32, 1)  y (32, 1)
  batch 3: x (32, 1)  y (32, 1)
  batch 4: x (32, 1)  y (32, 1)
  batch 5: x (32, 1)  y (32, 1)
  batch 6: x (8, 1)  y (8, 1)

the last batch holds 8, because 200 is not a multiple of 32


## `DataLoader` settings that matter {.smaller}

| Argument | What it does | What to set |
|---|---|---|
| `num_workers` | subprocesses that run `__getitem__` in parallel | **the one that matters.** Start at 4, raise while it still helps |
| `pin_memory` | stages batches in page-locked memory, so the copy to the GPU is faster and can overlap with compute | `True` whenever you train on a GPU |
| `persistent_workers` | keeps workers alive between epochs instead of respawning them | `True` when `num_workers > 0` |
| `prefetch_factor` | batches each worker prepares ahead of time | leave at 2 until profiling says otherwise |
| `shuffle` | reshuffles the order every epoch | `True` for training, `False` for validation and test |
| `drop_last` | discards the short final batch | `True` if you use batch norm, so you never get a batch of 1 |

```python
train_loader = DataLoader(train, batch_size=64, shuffle=True,
                          num_workers=4, pin_memory=True,
                          persistent_workers=True)
```

## Recognizing a data-bound training run {.smaller}

Run `nvidia-smi` while training. If GPU utilization keeps falling to 0% and
spiking back, the GPU is **waiting on data**. Raising `num_workers` will then
speed up the run far more than any change you make to the model.

Three things worth knowing before you hit them:

- On Windows and macOS, `num_workers > 0` starts workers by **spawning** new
  interpreters, which re-imports your script. Training code must sit behind
  `if __name__ == "__main__":` or it will spawn forever
- More workers is not always better. Each one holds its own copy of the
  dataset object, and past the number of CPU cores they compete rather than
  help
- If your examples have different lengths, batching them needs a `collate_fn`.
  The default one calls `torch.stack`, which requires identical shapes

::: {.callout-tip}
## Where to put the data on the GPU
`xb.to(device, non_blocking=True)` only overlaps the transfer with compute if
the loader used `pin_memory=True`. The two settings are meant to be used
together.
:::

## The loop, with batches

The five lines are unchanged. They now sit inside a second loop.

```python
for epoch in range(EPOCHS):
    model.train()
    for xb, yb in train_loader:          # one iteration per batch
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()

    model.eval()                          # dropout off, batch norm frozen
    with torch.no_grad():                 # no graph, since we will not
        val = sum(criterion(model(xb), yb).item()  #  differentiate this
                  for xb, yb in val_loader) / len(val_loader)
```

Note which things moved. `zero_grad` is **per batch**, not per epoch.
`model.train()` and `model.eval()` are **per phase**. The `no_grad` block wraps
evaluation only.

## Training, validation, and test {.smaller}

Three splits, three different jobs, and they are not interchangeable.

| Split | Used for | How often you look |
|---|---|---|
| **Training** | computing gradients | every step |
| **Validation** | choosing hyperparameters, early stopping | every epoch |
| **Test** | the one honest number | **once**, at the end |

The validation set is not a held-out set once you have used it to pick a
learning rate, a width, a dropout rate and a stopping epoch. You have fitted to
it, with you as the optimizer. That is why the test set exists and why it gets
looked at once.

```python
train, val = random_split(data, [0.8, 0.2],
                          generator=torch.Generator().manual_seed(6600))
```

## Leakage {.smaller}

Leakage is when information that should only be in the training set finds its
way into training anyway. The validation score then measures memorization
rather than generalization, so it looks excellent and the model is useless on
new data.

Four common ways it happens:

- **Normalizing before splitting.** The mean and standard deviation you
  subtracted contain the validation set. Fit the scaler on **train** and apply
  it to the others
- **Splitting randomly when the data is grouped.** Multiple rows per patient,
  per user or per document put the same entity on both sides
- **Splitting randomly when the data is a time series.** In production you only
  ever have the past, so training on future rows and testing on earlier ones
  measures something you will never be able to do. Split by time
- **Duplicates.** Near-identical rows on both sides are memorization dressed up
  as generalization

::: {.callout-important}
## How to spot leakage
Validation cost that is suspiciously close to training cost, or a jump in
performance right after you added a preprocessing step. Both are worth an hour
of checking.
:::

# Optimizers

- Why one learning rate is not enough for every parameter
- Momentum, RMSProp, Adam

## The problem with one learning rate {.smaller}

Plain SGD applies **one** step size to every parameter:

$$\theta \leftarrow \theta - \eta \nabla_{\theta} \mathcal{L}$$

The problem is that the cost surface is much steeper in some directions than in
others. A steep direction needs small steps, because a large one overshoots and
the cost goes up. A shallow direction needs large steps, because small ones
barely move. One value of $\eta$ has to work for both, so you are forced to
pick a value small enough for the steepest direction, and every shallow
direction then crawls.

The ratio of the steepest curvature to the shallowest is called the **condition
number**. The larger it is, the worse this mismatch gets, and real networks
have very large condition numbers.

Two independent fixes, and they compose:

- **Momentum**: accumulate a velocity, so consistent directions build speed
- **Adaptive rates**: give every parameter its own step size, from its own
  gradient history

## Momentum

Keep a running velocity instead of stepping on the raw gradient:

$$\mathbf{v} \leftarrow \beta \mathbf{v} + \nabla_{\theta}\mathcal{L}
\qquad
\theta \leftarrow \theta - \eta \mathbf{v}$$

$\beta$ is typically $0.9$. Two things happen at once:

- In a direction where the gradient keeps **pointing the same way**, the
  velocity accumulates and the effective step grows, up to $1/(1-\beta)$ times
  the plain step. At $\beta = 0.9$ that is a factor of 10
- In a direction where the gradient keeps **flipping sign**, successive terms
  cancel and the oscillation is damped

That is exactly the combination an ill-conditioned surface needs: speed along
the valley, calm across it.

## Momentum, measured {.smaller}

[![](images/fig-momentum.gif){width=1000 .dw92}](images/fig-momentum.gif){target="_blank" .zoom}

Both runs use the **same learning rate**, $0.005$, for 25 steps. SGD settles
into the floor of the valley almost immediately and then inches along it.
Momentum overshoots across the valley again and again, and that does not
matter, because the direction that counts is the shallow one and there it
keeps building speed.

After 25 steps SGD has covered $1.06$ along the shallow axis and momentum has
covered $5.94$, which is **5.6 times as far**. The right panel is the part a
still picture cannot show you: SGD's line is straight, momentum's bends upward.

## Why the momentum gap has that size

For a quadratic with curvatures $m$ and $L$, both optimal rates are known in
closed form, so the figure is not a tuning accident. Write
$\kappa = L/m$ for the condition number:

| | Best rate | Cost falls per step by |
|---|---|---|
| Plain SGD | $\eta = 2/(L+m)$ | $\dfrac{\kappa - 1}{\kappa + 1}$ |
| Momentum | $\eta = 4/(\sqrt{L}+\sqrt{m})^2$ | $\dfrac{\sqrt{\kappa} - 1}{\sqrt{\kappa} + 1}$ |

Momentum replaces $\kappa$ with $\sqrt{\kappa}$. At $\kappa = 100$ that is
$0.980$ per step against $0.818$ per step, which over 80 steps is the
difference between $0.2$ and $10^{-7}$.

That square root is the entire theoretical benefit of momentum, and the figure
is simply that formula playing out.

## RMSProp and Adam

**RMSProp** divides each parameter's step by the running root-mean-square of
its own recent gradients:

$$s \leftarrow \rho s + (1-\rho)g^2
\qquad
\theta \leftarrow \theta - \frac{\eta}{\sqrt{s} + \epsilon}\, g$$

A parameter with consistently large gradients gets a smaller step; one with
tiny gradients gets a larger step. The learning rate becomes **per parameter**
without you choosing millions of them.

**Adam** is momentum and RMSProp at the same time, plus a correction for the
fact that both running averages start at zero:

$$m \leftarrow \beta_1 m + (1-\beta_1) g
\qquad
v \leftarrow \beta_2 v + (1-\beta_2) g^2$$

$$\hat{m} = \frac{m}{1-\beta_1^t}, \quad \hat{v} = \frac{v}{1-\beta_2^t}
\qquad
\theta \leftarrow \theta - \frac{\eta}{\sqrt{\hat{v}} + \epsilon}\hat{m}$$

Defaults $\beta_1 = 0.9$, $\beta_2 = 0.999$, $\epsilon = 10^{-8}$ are
almost never worth changing.

## Four optimizers on the same problem {.smaller}

[![](images/fig-optimizers.png){width=1000 .dw92}](images/fig-optimizers.png){target="_blank" .zoom}

The same quadratic, condition number 100, 80 steps, and **each method running
at its own best learning rate**, so none of them is handicapped to make a
point. SGD never gets within $0.05$ of the minimum and ends at cost $4.59$.
Momentum arrives at **step 39** and ends at $1.8 \times 10^{-8}$. RMSProp
arrives at step 55, Adam at step 78.

Do not read too much into that ordering. This is a clean quadratic and real
loss surfaces are not. What Adam actually buys you is not the best final
number, it is getting close to the answer **without you having to find the
best learning rate first**, which is exactly the position you are in when you
start.

## Optimizers in PyTorch {.smaller}

| Optimizer | Starting rate | When you would choose it |
|---|---|---|
| `SGD(lr=0.1)` | $10^{-1}$ | Almost never on its own. It is the baseline everything else is compared against |
| `SGD(lr=0.1, momentum=0.9)` | $10^{-1}$ | Image models, once you are willing to tune a learning-rate schedule. Often reaches the best final accuracy |
| `RMSprop(lr=1e-3)` | $10^{-3}$ | You will meet it in older recurrent-network code. Adam has replaced it for new work |
| `Adam(lr=1e-3)` | $10^{-3}$ | **Start here.** It works without tuning, which is what you want before you trust your code |
| `AdamW(lr=1e-3)` | $10^{-3}$ | Start here instead whenever you want weight decay. Same as Adam, with the decay implemented correctly |

```python
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
```

Note the rates differ by **two orders of magnitude** between SGD and Adam. A
learning rate copied from an SGD recipe into Adam destabilizes training: on the
Lab 4 network, Adam at $0.1$ instead of $10^{-3}$ drops validation accuracy
from $0.85$ to $0.62$. Nothing raises, which is what makes this one of the most
common ways a working script stops working.

## Choosing an optimizer

The honest summary:

1. **Start with Adam or AdamW at $10^{-3}$.** It is the least sensitive to a
   bad learning rate, which is what you want when you do not yet know whether
   your model is even wired up correctly
2. **If you are reproducing a paper, use what the paper used.** This is not
   deference, it is that the learning rate and schedule were tuned together
3. **Well-tuned SGD with momentum often generalizes slightly better** than Adam
   on vision tasks. It also takes much more tuning to get there
4. **The optimizer is rarely your problem.** If the model is not learning, the
   cause is far more often the data, the shapes, the learning rate or the
   initialization

::: {.callout-note}
## What to change first
Learning rate, by factors of ten, before anything else. It is the single most
important hyperparameter, and it is not close.
:::

# Learning-rate schedules

- The right rate at step 1 is the wrong rate at step 10,000
- Warmup, decay, and clipping

## Why the rate should change

Early in training you are far from any minimum and want to cover ground. Late
in training you are near one and want to stop overshooting it. A **schedule**
is a rule for lowering $\eta$ over time.

You can recognize a rate that should have been lowered. The cost falls for a
while, then flattens into a noisy band and stops improving. What is happening
is that the optimizer is stepping over the minimum and back again. Cut the rate
and the cost drops visibly within a few epochs, which is why loss curves under
a step schedule have that staircase shape.

## Five schedules

[![](images/fig-lr-schedules.png){width=1000 .dw89}](images/fig-lr-schedules.png){target="_blank" .zoom}

All five start at $\eta = 0.1$. At epoch 50, step decay is at $0.0100$,
exponential is at $0.0077$, and cosine is at $0.0500$.

**Cosine annealing** is the common default: smooth, no cliff edges, and it ends
near zero without you choosing when.

## Warmup

The first few hundred steps are the most dangerous in a training run. The
weights are random, so the gradients are large and unrepresentative, and
Adam's variance estimate $\hat{v}$ is built from almost no data.

**Warmup** ramps the rate linearly from near zero to the target over a few
hundred steps or a few epochs, then hands over to the main schedule.

- Essential for transformers, where training without it frequently diverges in
  the first thousand steps
- Cheap insurance everywhere else
- Shows up in the figure as the green line climbing before it anneals

Usually a few percent of total training. Longer does no harm.

## Gradient clipping {.smaller}

Clipping puts a ceiling on how large one update can be. If a single batch
produces an enormous gradient, one step can destroy a model that took hours to
train.

[![](images/fig-grad-clipping.png){width=1000 .dw85}](images/fig-grad-clipping.png){target="_blank" .zoom}

**Left.** A gradient already below the threshold $C$ passes through untouched.
Anything above $C$ is scaled down to exactly $C$.

**Middle.** Two versions, and they are not the same operation.
`clip_grad_norm_` rescales the whole vector and the direction does not change.
`clip_grad_value_` clamps each component on its own, moving this gradient 14
degrees. Use the norm version.

**Right.** The same run twice, with one corrupted batch at step 14. Unclipped,
that step travels 24 units, lands at $(21, -5)$ and takes the cost from $2.3$
to $443$. Clipped, it heads the same way and travels $1.3$, and the run ends
at $0.12$.

## Gradient clipping in the training loop {.smaller}

Every parameter gradient in the model is treated as **one long vector**
$\mathbf{g}$. The norm is taken across the whole model, not per layer. Then:

$$\mathbf{g} \leftarrow C \cdot \frac{\mathbf{g}}{\|\mathbf{g}\|}
\qquad \text{whenever} \qquad \|\mathbf{g}\| > C$$

```python
loss.backward()                                    # gradients now exist
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
optimizer.step()                                   # and are used here
```

Those three lines have to be in that order. Clipping needs the gradients to
exist, so it goes after `backward()`, and it has to act before `step()` reads
them.

- $C$ is usually between **1 and 10**. Start at $1.0$
- It is standard practice for recurrent networks and transformers. Everywhere
  else it costs almost nothing and occasionally saves a run
- If clipping fires on most steps, $C$ is too low. At that point you are
  ignoring the size of the gradient entirely and following only its direction

## Schedules in PyTorch

```python
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

for epoch in range(EPOCHS):
    for xb, yb in train_loader:
        optimizer.zero_grad()
        criterion(model(xb), yb).backward()
        optimizer.step()
    scheduler.step()                     # once per epoch, not per batch
```

`scheduler.step()` goes **outside** the batch loop for epoch-based schedules
and inside it for step-based ones. Putting an epoch-based scheduler inside the
batch loop decays the rate $\lceil N/B \rceil$ times too fast, which looks
like a model that learns for one epoch and then stops.

Worth printing: `optimizer.param_groups[0]["lr"]`.

# Initialization

- Initialization decides whether a deep network trains at all
- Xavier and He

## Why initialization decides whether training starts

In Week 2 we set the weights randomly and moved on. At two layers it did not
matter. At twenty layers it decides whether the run does anything at all.

The forward pass multiplies by a weight matrix at every layer, and the backward
pass multiplies by its transpose. Both are products of many factors:

- If the factors are **slightly above 1**, twenty of them multiply to something
  enormous
- If they are **slightly below 1**, twenty of them multiply to nothing
- Only a narrow band keeps the signal roughly the same size all the way through

Zero initialization is worse still: every unit in a layer computes the same
thing, receives the same gradient, and updates identically forever. A 512-unit
layer does the work of one unit. **Randomness is what breaks that symmetry.**

## Initialization, measured {.smaller}

[![](images/fig-init-variance.png){width=1000 .dw92}](images/fig-init-variance.png){target="_blank" .zoom}

Twelve tanh layers, 256 wide. **Only the initialization differs**, and the two
panels disagree about who is in trouble.

**Left**: $\mathcal{N}(0,1)$ looks fine, holding an activation spread near
$1.0$ the whole way down. **Right**: it is the worst of the four. Gradient
norms run from $9.9 \times 10^{2}$ at layer 1 to $7.2 \times 10^{-3}$ at
layer 12, five orders of magnitude apart.

## Why the two naive initializations fail {.smaller}

**$\mathcal{N}(0,1)$, the red line.** With 256 inputs, the pre-activations
have a standard deviation near 16, so tanh is pinned against $\pm 1$:
**86.6%** of units in the last layer are saturated. The activations look
healthy precisely *because* they are clipped. A saturated tanh has derivative
$1 - a^2 \approx 0$, but the weights are large enough that the backward
multiplication more than makes up for it, so the gradient **explodes** as it
travels back toward the input.

**$\mathcal{N}(0, 0.01)$, the blue line.** The signal is gone by layer 2, and
the gradient is around $10^{-19}$ at every layer. Nothing will ever update.

One is far too large and one is far too small, and the forward pass tells you
this about neither of them.

## Why Xavier and He both work {.smaller}

**Xavier, the green line.** Activation spread settles near $0.21$, and the
gradient norm is $1.3 \times 10^{-2}$ at layer 1 against
$1.7 \times 10^{-2}$ at layer 12. That is flat, and flat is what you want.

**He, the orange line.** Also flat: $3.6 \times 10^{-2}$ at layer 1 against
$4.4 \times 10^{-2}$ at layer 12. He uses a larger variance than Xavier,
$2/n_{\text{in}}$ rather than $2/(n_{\text{in}} + n_{\text{out}})$, so its
activations sit higher, near $0.56$ instead of $0.21$, and $0.1\%$ of units
saturate rather than none. That is slightly more than a tanh network needs,
and it trains perfectly well anyway.

**Both principled schemes work and both naive ones fail**, and the gap between
those two groups is twenty orders of magnitude. Matching the scheme to the
activation function is a refinement. Using one at all is not.

When you are debugging, remember that **a forward pass that looks reasonable
proves nothing.** Check the gradients.

## Xavier and He {.smaller}

Both pick the variance of $\mathbf{W}$ so that the signal keeps its size as it
passes through a layer. They differ in which activation they assume.

| | Variance | For |
|---|---|---|
| **Xavier / Glorot** | $\dfrac{2}{n_{\text{in}} + n_{\text{out}}}$ | tanh, sigmoid |
| **He / Kaiming** | $\dfrac{2}{n_{\text{in}}}$ | ReLU and its relatives |

He carries the extra factor of 2 because ReLU zeroes half its inputs, so half
the variance is lost and has to be put back.

```python
for m in model.modules():
    if isinstance(m, nn.Linear):
        nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
        nn.init.zeros_(m.bias)
```

**Biases start at zero.** The weights already break the symmetry, so there is
nothing left for a random bias to fix.

`nn.Linear` already defaults to a Kaiming-uniform variant, which is why the
small models in this course train without you touching any of this. Past about
ten layers it stops being automatic.

# Regularization

- Fitting the noise, and four ways to stop

## Overfitting, measured {.smaller}

[![](images/fig-overfitting.png){width=1000 .dw92}](images/fig-overfitting.png){target="_blank" .zoom}

Forty noisy points, a 33,000-parameter network, 3000 epochs of Adam. The dotted
line is the **noise floor**, $0.22^2 = 0.048$: no honest model can score below
it, because that much of the target is noise.

**Left**: training cost goes **through** the floor to $0.015$, which is the
definition of fitting the noise. Validation bottoms out at epoch **104** and
then gets $1.81$ times worse.

## Reading the gap

The two curves separating is the entire diagnosis. Training cost falling while
validation cost rises means the model is learning things that are true of
**these 40 points** and false of everything else.

Three numbers from that run worth carrying:

- Validation was at its best after **104 of 3000 epochs**. The remaining 97% of
  the compute made the model worse
- Training cost ended at $0.015$, **below** the $0.048$ noise floor. A model
  that beats the noise floor on training data has memorized the noise
- With dropout $0.1$ and weight decay $10^{-3}$, the final model is **1.77
  times better** on validation, and the validation curve never turns around

::: {.callout-note}
## Why a noise floor is worth computing
If you know roughly how noisy your labels are, you know what a good cost looks
like. A training cost far below that is not a triumph.
:::

## Early stopping

The cheapest regularizer, and the one you should always have on. Evaluate on
validation every epoch, keep the best weights, and stop when they stop
improving.

```python
best, best_state, patience, waited = float("inf"), None, 20, 0
for epoch in range(EPOCHS):
    train_one_epoch()
    val = evaluate()
    if val < best:
        best, best_state, waited = val, copy.deepcopy(model.state_dict()), 0
    else:
        waited += 1
        if waited >= patience:
            break
model.load_state_dict(best_state)         # the point of the whole exercise
```

Two details people skip. **Patience**, so one noisy epoch does not stop the
run. And **restoring the best weights**, without which you have carefully
identified the best epoch and then shipped a later one.

## Weight decay

Add a penalty on the size of the weights to the cost:

$$\mathcal{L}_{\text{total}} = \mathcal{L} + \lambda \sum_j w_j^2$$

The gradient of the penalty is $2\lambda w$, so every step pulls every weight
a little toward zero. Large weights have to **earn** their size by reducing the
data term more than the penalty costs.

This is ridge, or $L_2$, regularization, arriving in a new setting. $L_1$
penalties, $\lambda \sum_j |w_j|$, drive weights to exactly zero instead and
are used when sparsity is the goal.

```python
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
```

::: {.callout-important}
## `Adam(weight_decay=...)` is not `AdamW`
In plain `Adam` the penalty goes through the adaptive rescaling, so parameters
with small gradients get decayed harder. `AdamW` applies the decay separately,
which is what the maths above describes. **Use `AdamW`.**
:::

## Dropout

[![](images/fig-dropout.png){width=960 .dw89}](images/fig-dropout.png){target="_blank" .zoom}

At every training step, switch off each hidden unit independently with
probability $p$. A different subnetwork is trained each step, and no unit can
rely on any particular other unit being present.

At evaluation, all units are active and the activations are rescaled so the
expected value matches training.

## Dropout in practice

```python
self.net = nn.Sequential(
    nn.Linear(256, 256), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(256, 256), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(256, 10),                      # never after the output layer
)
```

- $p$ between $0.1$ and $0.5$. Start at $0.2$
- **After** the activation, not before
- **Not** after the final layer. You would be randomly deleting predictions
- `model.eval()` is what switches it off. This is the reason that call exists,
  and forgetting it makes your validation number both worse and unrepeatable

::: {.callout-note}
## Why dropout works
Dropout is an ensemble argument. You are training exponentially many
subnetworks that share weights, and evaluating something close to their
average. It also acts like noise injection, which discourages the model from
depending on any single feature.
:::

## Batch and layer normalization {.smaller}

Standardize the activations **inside** the network, not just the inputs.

**Batch norm** normalizes each feature across the batch, then applies a learned
scale and shift:

$$\hat{z} = \frac{z - \mu_{\text{batch}}}{\sqrt{\sigma^2_{\text{batch}} + \epsilon}},
\qquad y = \gamma \hat{z} + \beta$$

It lets you use larger learning rates, reduces sensitivity to initialization,
and regularizes slightly as a side effect of the batch noise.

**The catch** is that it behaves differently in training and evaluation. In
training it uses this batch's statistics; in evaluation it uses a running
average collected during training. That is the other half of why
`model.eval()` exists, and why batch norm is unreliable at very small batch
sizes.

**Layer norm** normalizes across the features of each example instead, so it
does not depend on the batch at all. That is why transformers use it.

## What to reach for, in order {.smaller}

Overfitting is not one problem with one fix. In rough order of what to try:

1. **More data.** Nothing else comes close. Augmentation counts
2. **Early stopping.** Free, and you should already have it
3. **Weight decay.** One number, usually $10^{-4}$ to $10^{-2}$
4. **Dropout.** Start at $0.2$ in the wide layers
5. **A smaller model.** Effective, and the last resort, because capacity you
   remove is capacity you cannot get back by tuning

::: {.callout-important}
## Diagnose before you treat
Training cost **and** validation cost both high is **underfitting**. More
regularization will make it worse. Train longer, use a bigger model, or check
that your learning rate is not wrong.

Regularization is for the case where training cost is low and validation cost
is not.
:::

# Putting it all together

- Everything in this deck, in one script

## The full loop {.smaller}

Every line here has appeared somewhere above. This is the shape of every
training script for the rest of the semester.

```python
model = MLP().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.MSELoss()

for epoch in range(EPOCHS):
    model.train()                                  # dropout on
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()                      # clear the running sum
        loss = criterion(model(xb), yb)            # forward
        loss.backward()                            # backward
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()                           # update
    scheduler.step()                               # once per epoch

    model.eval()                                   # dropout off
    with torch.no_grad():                          # no graph
        val = mean(criterion(model(xb.to(device)), yb.to(device)).item()
                   for xb, yb in val_loader)
    if val < best:
        best, best_state, waited = val, deepcopy(model.state_dict()), 0
    else:
        waited += 1
        if waited >= PATIENCE:
            break

model.load_state_dict(best_state)
```

## Saving a model

Save the **`state_dict`**, not the model object.

```python
torch.save({"model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "epoch": epoch}, "checkpoint.pt")

ckpt = torch.load("checkpoint.pt", map_location=device)
model.load_state_dict(ckpt["model"])
```

`torch.save(model)` pickles your class definition along with the weights, so
the file breaks when you rename or move the class. A `state_dict` is just a
dictionary of tensors and will still load in a year.

Save the **optimizer** state too if you intend to resume. Adam's $m$ and $v$
took the whole run to build, and restarting without them gives you a visible
bump in the cost curve.

## Reproducibility

```python
torch.manual_seed(SEED)
np.random.seed(SEED)
generator = torch.Generator().manual_seed(SEED)       # for DataLoader shuffling
loader = DataLoader(data, batch_size=32, shuffle=True, generator=generator)
```

Seeding `torch` alone is not enough: the `DataLoader`'s shuffling, any
augmentation, and dropout masks all draw from streams you have to seed
explicitly.

Even then, bit-exact reproducibility across **different GPUs** or different
library versions is not something to expect. What you should expect is that the
same script on the same machine gives the same curve, and that is enough to
tell a real improvement from noise.

::: {.callout-note}
## Why seeding matters
Without a seed you cannot tell whether your change helped or whether you got a
better draw. Run the baseline twice before believing any small improvement.
:::

## When a model will not learn {.smaller}

In the order that finds the problem fastest:

1. **Can it overfit 10 examples?** Take a tiny subset, turn off
   regularization, and drive the training cost to near zero. If it cannot, the
   bug is in the code, not the hyperparameters. This is the highest-value test
   in the list
2. **Print shapes.** Especially `pred.shape` against `y.shape`
3. **Is the cost even changing?** If not: missing `optimizer.step()`, a
   learning rate of zero, parameters not registered, or `requires_grad` off
4. **Is the cost `nan`?** Learning rate too high, a $\log$ of zero, or a
   division by a near-zero. Clip, and lower the rate
5. **Learning rate, by factors of ten.** Both directions
6. **Are `zero_grad` and `train`/`eval` in the right places?**
7. **Look at the data.** Plot a batch. Check the labels line up with the
   inputs. Check the scaling
8. **Only then** change the architecture

::: {.callout-important}
## Overfit a tiny subset first
Overfit a tiny subset **first**, every time, before any real run. It separates
"my code is wrong" from "my model is wrong," and those have completely
different fixes.
:::

# Closing

- Assessment
- Supplemental content
- Quiz 4, right now

## Labs and quizzes {.smaller}

**Lab 3 was due yesterday**, Wednesday Sep 23. It was the backpropagation lab:
the `Value` class, the matrix backward pass, and the forward-pass count.

**Quiz 4 is today**, at the end of class. It covers **last week's
backpropagation material plus Lab 3**: the worked backward pass, summing over
paths, topological order, reverse against forward mode, the sigmoid and
cross-entropy cancellation, the three layer gradients, and vanishing
gradients. **Nothing from this deck is on it.**

::: {.callout-note}
## When this deck becomes quizzable
Quiz 5, next week. PyTorch, the training loop, optimizers, schedules,
initialization and regularization. Everything from today, plus Lab 4.

**The format changes for this one**, because the material is code. Instead of
the multiple-answer and short-answer parts you have seen, there is a
fill-in-the-blank part where you compute a number, and a part where you are
given a training script that runs fine and trains badly, and you find three
things wrong with it. The study guide goes through all of it.
:::

## Supplemental content {.smaller}

These are **optional**. No graded work assumes you read them. They go deeper
than we have time for, and the PyTorch pages are the source material this deck
was built from.

:::: {.columns}
::: {.column width="50%"}
**PyTorch**

- [Tensors](https://jfh.georgetown.domains/centralized-lecture-content/content/machine-learning/deep-learning/pytorch/torch-tensors/notes.html)
- [Models](https://jfh.georgetown.domains/centralized-lecture-content/content/machine-learning/deep-learning/pytorch/torch-models/torch-models.html)
- [Data](https://jfh.georgetown.domains/centralized-lecture-content/content/machine-learning/deep-learning/pytorch/torch-data/torch-data.html)
:::
::: {.column width="50%"}
**Optimization and training**

- [Gradient descent](https://jfh.georgetown.domains/centralized-lecture-content/content/machine-learning/deep-learning/fundamentals/gradient-descent/notes.html)
- [Numerical optimization](https://jfh.georgetown.domains/centralized-lecture-content/content/mathematics/solvers-and-optimization/numerical-methods/numerical-optimization/notes.html)
- [Automatic differentiation](https://jfh.georgetown.domains/centralized-lecture-content/content/machine-learning/deep-learning/backprop/automatic-differentiation/automatic-differentiation.html)
:::
::::

<sup>All of these live on the
[centralized lecture content](https://jfh.georgetown.domains/centralized-lecture-content/)
site. The Data page covers `Dataset` and `DataLoader` in more detail than this
deck does, including the multiprocessing options we skipped, and is the one to
read before the next lab.</sup>

## Wrap-up {.smaller}

The gradient stopped being the hard part three weeks ago. What is left:

- A **tensor** is an array that records the operations producing it, which is
  what makes `.backward()` possible
- **`.grad` accumulates**, so `zero_grad()` is not optional, and the reason is
  the fork in the graph from Week 4
- **`nn.Module`** registers parameters; **`model.eval()`** switches dropout off
  and freezes batch norm, and forgetting it corrupts your validation number
- **Mini-batch noise** falls like $1/\sqrt{B}$, and it is useful rather than
  merely tolerable
- **Momentum** turns $\kappa$ into $\sqrt{\kappa}$; **Adam** gives every
  parameter its own rate and is the right first choice
- **Initialization** decides whether a deep network trains at all, and a
  healthy forward pass proves nothing about the backward pass
- **Regularization** is for low training cost with high validation cost, and
  for nothing else

::: {.callout-note}
## Next
**Convolutional networks.** Everything here still applies; what changes is the
layer. Weight sharing, locality, and why a fully connected layer is the wrong
tool for an image.
:::

## Quiz 4 {.smaller}

Closed-book, no notes. A formula sheet comes with it.

Covers **the second half of backpropagation**, picking up where Quiz 3
stopped:

- Local derivatives at a node: add, multiply, ReLU
- Working a backward pass through a graph and reading off the numbers
- Adding gradients where two paths meet
- Topological order, and what breaks without it
- Reverse mode against forward mode, and why a scalar cost decides it
- The sigmoid and cross-entropy cancellation
- The three gradients a layer owes, and why nobody forms a Jacobian
- Exploding and vanishing gradients

**Plus Lab 3**: why `Value` accumulates with `+=`, why the backward pass needs
a seed of $1.0$, and what 20,001 against 401 showed.

Twenty-five points, about fifteen minutes. Nothing from today's deck.